# 🤖 Mini-projet : Intégration MCP + Agents multi-serveurs avec Gemini

**Objectif :** Créer une application **multi-agents** complète qui orchestre plusieurs **serveurs MCP** (Model Context Protocol) à l'aide d'un LLM (**Gemini**) avec une **politique flexible pilotée par outil** : c'est le LLM qui détermine les prochaines étapes, pas un flux prédéfini.

**Thème :** *Assistant d'espace de travail* (fichiers + git + résumé personnalisé)

**Architecture :**

```
                    ┌──────────────────────┐
                    │   SUPERVISEUR (LLM)  │  ← Gemini décide dynamiquement
                    │  politique par outil │     quel agent appeler
                    └─────────┬────────────┘
          ┌───────────────────┼───────────────────┐
          ▼                   ▼                   ▼
  ┌───────────────┐   ┌──────────────┐   ┌────────────────┐
  │ Agent Fichiers│   │  Agent Git   │   │ Agent Analyse  │
  │ (ReAct+Gemini)│   │ (ReAct+Gemini)│  │ (ReAct+Gemini) │
  └───────┬───────┘   └──────┬───────┘   └───────┬────────┘
          ▼                  ▼                   ▼
  ┌───────────────┐   ┌──────────────┐   ┌────────────────┐
  │ MCP filesystem│   │  MCP git     │   │ MCP custom_ops │
  │ (tiers, npx)  │   │ (tiers, py)  │   │ (FastMCP perso)│
  └───────────────┘   └──────────────┘   └────────────────┘
```

**Serveurs MCP utilisés :**
1. `@modelcontextprotocol/server-filesystem` — serveur **tiers** (Node/npx)
2. `mcp-server-git` — serveur **tiers** (Python)
3. `custom_ops` — serveur **personnalisé** écrit avec **FastMCP**


---
## 1️⃣ Configuration de Colab : installer les dépendances

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "mcp-server-git" \
  "fastmcp>=2.0.0"

print("✅ Dépendances installées")

**Notes :**
* `langchain-google-genai` est l'intégration LangChain pour Gemini.
* `langchain-mcp-adapters` fournit un client MCP compatible avec les outils LangChain.
* `mcp-server-git` est le serveur MCP tiers officiel pour Git (Python).
* `fastmcp` servira à créer notre serveur MCP personnalisé (étape 8).

---
## 2️⃣ Configurer `GOOGLE_API_KEY`

Deux options :
1. **Recommandé (Colab)** : ajoutez votre clé dans les *Secrets* Colab (icône 🔑 à gauche) sous le nom `GOOGLE_API_KEY`.
2. **Fallback** : saisie manuelle sécurisée via `getpass`.

👉 Obtenez une clé gratuite sur [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
import os

# Option 1 : secrets Colab
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ Clé chargée depuis les secrets Colab")
except Exception:
    # Option 2 : saisie manuelle (fonctionne aussi en local)
    if not os.environ.get("GOOGLE_API_KEY"):
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Entrez votre GOOGLE_API_KEY : ")
    print("✅ Clé configurée manuellement")

assert os.environ.get("GOOGLE_API_KEY"), "❌ GOOGLE_API_KEY manquante !" 

---
## 3️⃣ Vérifier la disponibilité de Node / npm / npx

De nombreux serveurs MCP sont distribués sous forme de packages Node exécutables via `npx` (c'est le cas du serveur `filesystem`).

In [ ]:
!node --version
!npx --version

In [ ]:
# ⚠️ À exécuter UNIQUEMENT si la cellule précédente échoue (Node absent)
import shutil
if shutil.which("node") is None or shutil.which("npx") is None:
    !apt-get -qq update
    !apt-get -qq install -y nodejs npm
    !node --version
    !npx --version
else:
    print("✅ Node et npx déjà disponibles, rien à faire.")

---
## 4️⃣ Préparer l'espace de travail (WORKDIR)

Notre « Assistant d'espace de travail » a besoin d'un répertoire avec :
* quelques fichiers d'exemple,
* un dépôt **git** initialisé (avec un historique + des modifications non commitées, pour rendre les démos intéressantes).

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

WORKDIR = "/content/workspace"
os.makedirs(WORKDIR, exist_ok=True)

# --- Fichiers d'exemple ---
(Path(WORKDIR) / "README.md").write_text(textwrap.dedent('''\
    # Projet Demo
    Un petit projet de démonstration pour l'assistant d'espace de travail.

    ## Fonctionnalités
    - Calculs simples
    - Gestion de tâches
'''), encoding="utf-8")

(Path(WORKDIR) / "app.py").write_text(textwrap.dedent('''\
    def add(a, b):
        return a + b

    def multiply(a, b):
        return a * b

    if __name__ == "__main__":
        print("2 + 3 =", add(2, 3))
'''), encoding="utf-8")

(Path(WORKDIR) / "TODO.txt").write_text(
    "- corriger le bug d'affichage\n- ajouter des tests\n\n- écrire la doc\n",
    encoding="utf-8")

# --- Initialiser git + premier commit ---
def git(*args):
    return subprocess.run(["git", "-C", WORKDIR, *args],
                          capture_output=True, text=True).stdout

git("init")
git("config", "user.email", "demo@example.com")
git("config", "user.name", "Demo User")
git("add", "-A")
git("commit", "-m", "Commit initial : structure du projet")

# --- Une modification NON commitée (pour la démo du changelog) ---
with open(Path(WORKDIR) / "app.py", "a", encoding="utf-8") as f:
    f.write("\ndef subtract(a, b):\n    return a - b\n")

print("✅ Espace de travail prêt :", os.listdir(WORKDIR))
print(git("status", "--short"))

---
## 5️⃣ & 6️⃣ Lancer / connecter les serveurs MCP tiers (transport `stdio`)

Dans MCP, l'agent communique avec les serveurs via **stdio** : le client lance des **sous-processus** (`npx -y <package>` ou `python -m <module>`), et échange des messages JSON-RPC sur stdin/stdout.

Nous enregistrons **deux serveurs MCP tiers** avec `MultiServerMCPClient` :
1. **filesystem** (Node, via `npx`) — lecture/écriture de fichiers dans `WORKDIR`
2. **git** (Python, via `python -m mcp_server_git`) — statut, diff, log, commit…

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()   # indispensable dans Colab/Jupyter (event loop déjà actif)

from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
}

# tool_name_prefix=True → les outils sont préfixés par le nom du serveur
# (ex. "filesystem_read_file"), ce qui évite les collisions de noms.
try:
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
except TypeError:
    # Selon la version de langchain-mcp-adapters, le paramètre peut ne pas exister
    client = MultiServerMCPClient(mcp_connections)

tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print(f"✅ {len(tools)} outils MCP découverts :\n")
for t in tools:
    print(f"  • {t.name}")

---
## 7️⃣ Créer un agent Gemini capable d'utiliser ces outils

Nous utilisons `create_react_agent` de **LangGraph** : une boucle **ReAct** dans laquelle Gemini décide lui-même quels outils appeler, dans quel ordre, et quand s'arrêter — c'est déjà une politique *pilotée par outil* au niveau d'un agent.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Agent "généraliste" : accès à TOUS les outils MCP tiers
workspace_agent = create_react_agent(
    llm,
    tools,
    prompt=(
        "Tu es un assistant d'espace de travail. Tu as accès à des outils MCP "
        "pour manipuler les fichiers et le dépôt git du répertoire "
        f"{WORKDIR}. Utilise les outils dès que nécessaire, puis réponds en français "
        "de façon concise. Ne réponds jamais de mémoire sur le contenu des fichiers : "
        "lis-les avec les outils."
    ),
)

async def run_agent(agent, query: str):
    """Exécute un agent et affiche la trace des appels d'outils + la réponse."""
    result = await agent.ainvoke({"messages": [("user", query)]})
    for msg in result["messages"]:
        calls = getattr(msg, "tool_calls", None)
        if calls:
            for c in calls:
                print(f"  🔧 Appel outil → {c['name']}({c['args']})")
    print("\n💬 Réponse finale :\n")
    print(result["messages"][-1].content)
    return result

print("✅ Agent Gemini prêt")

### 🧪 Test 1 — Fichiers (serveur MCP tiers `filesystem`)

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(run_agent(
    workspace_agent,
    "Liste les fichiers de l'espace de travail, puis lis README.md et résume-le en une phrase."
))

### 🧪 Test 2 — Git (serveur MCP tiers `git`)

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(run_agent(
    workspace_agent,
    "Quel est le statut git du dépôt ? Y a-t-il des modifications non commitées ? "
    "Si oui, montre le diff et explique ce qui a changé."
))

---
## 8️⃣ Implémenter un serveur MCP **personnalisé** en Python (FastMCP)

Notre serveur `custom_ops` expose des outils adaptés au thème « assistant d'espace de travail » :

| Outil | Rôle |
|---|---|
| `ping()` | Health check |
| `summarize_lines(lines)` | Statistiques sur une liste de lignes |
| `format_markdown(title, sections)` | Génère un rapport Markdown propre |
| `generate_changelog(diff)` | Transforme un `git diff` en changelog structuré |

In [ ]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_mcp_server.py")
server_path.write_text(textwrap.dedent('''
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        """Health check tool."""
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        """Retourne des statistiques sur une liste de lignes de texte
        (nombre total, lignes non vides, nombre de mots)."""
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        words = sum(len(l.split()) for l in lines)
        return {"total_lines": total, "nonempty_lines": nonempty, "word_count": words}

    @mcp.tool
    def format_markdown(title: str, sections: Dict[str, str]) -> str:
        """Génère un document Markdown propre à partir d'un titre et d'un
        dictionnaire {nom_de_section: contenu}."""
        out = [f"# {title}", ""]
        for name, content in sections.items():
            out.append(f"## {name}")
            out.append("")
            out.append(content.strip())
            out.append("")
        return "\n".join(out)

    @mcp.tool
    def generate_changelog(diff: str) -> str:
        """Transforme la sortie d'un `git diff` en changelog structuré :
        liste les fichiers modifiés, les lignes ajoutées (+) et supprimées (-)."""
        files, added, removed = [], [], []
        for line in diff.splitlines():
            if line.startswith("+++ b/"):
                files.append(line[6:])
            elif line.startswith("+") and not line.startswith("+++"):
                added.append(line[1:].strip())
            elif line.startswith("-") and not line.startswith("---"):
                removed.append(line[1:].strip())
        out = ["# Changelog", "", "## Fichiers modifiés"]
        out += [f"- `{f}`" for f in files] or ["- (aucun)"]
        out += ["", f"## Ajouts ({len(added)} lignes)"]
        out += [f"+ {a}" for a in added if a][:20] or ["- (aucun)"]
        out += ["", f"## Suppressions ({len(removed)} lignes)"]
        out += [f"- {r}" for r in removed if r][:20] or ["- (aucune)"]
        return "\n".join(out)

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''), encoding="utf-8")

print("✅ Wrote:", server_path)

---
## 9️⃣ Ajouter le serveur personnalisé au client MCP

On reconstruit un client avec **les 3 serveurs** (2 tiers + 1 personnalisé).

In [ ]:
mcp_connections["custom_ops"] = {
    "transport": "stdio",
    "command": "python",
    "args": [str(server_path)],
}

try:
    client2 = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
except TypeError:
    client2 = MultiServerMCPClient(mcp_connections)

tools2 = asyncio.get_event_loop().run_until_complete(client2.get_tools())

print("Tool count:", len(tools2))
print("\nOutils du serveur personnalisé :")
print([t.name for t in tools2 if "custom_ops" in t.name or t.name in
       ("ping", "summarize_lines", "format_markdown", "generate_changelog")])

### 🧪 Test 3 — Le serveur personnalisé répond-il ?

In [ ]:
# Petit test direct : appeler l'outil ping sans passer par le LLM
ping_tool = next(t for t in tools2 if t.name.endswith("ping"))
res = asyncio.get_event_loop().run_until_complete(ping_tool.ainvoke({}))
print("ping →", res)

---
## 🔟 Architecture MULTI-AGENTS : superviseur + agents spécialisés

C'est le cœur du projet. Nous créons :

* **3 agents spécialisés** (ReAct + Gemini), chacun connecté aux outils d'**un seul** serveur MCP ;
* **1 superviseur** (Gemini) qui voit ces agents comme des **outils** et décide **dynamiquement** — à chaque étape — quel agent appeler, avec quelle instruction, et quand la tâche est terminée.

👉 Aucun flux n'est codé en dur : **la politique est entièrement pilotée par le LLM via les appels d'outils** (*tool-driven policy*).

In [ ]:
from langchain_core.tools import tool

# --- Répartir les outils MCP par serveur ---
def tools_of(server: str):
    return [t for t in tools2 if t.name.startswith(server)]

fs_tools     = tools_of("filesystem") or [t for t in tools2 if "file" in t.name or "director" in t.name]
git_tools    = tools_of("git")        or [t for t in tools2 if t.name.startswith("git_")]
custom_tools = tools_of("custom_ops") or [t for t in tools2 if t.name in
                 ("ping", "summarize_lines", "format_markdown", "generate_changelog")]

print(f"filesystem: {len(fs_tools)} outils | git: {len(git_tools)} outils | custom_ops: {len(custom_tools)} outils")

# --- 3 agents spécialisés ---
fichiers_agent = create_react_agent(
    llm, fs_tools,
    prompt=(f"Tu es l'AGENT FICHIERS. Tu manipules les fichiers de {WORKDIR} "
            "avec tes outils MCP filesystem (lister, lire, écrire, créer). "
            "Réponds en français, de façon factuelle, en citant le contenu lu."),
)

git_agent = create_react_agent(
    llm, git_tools,
    prompt=(f"Tu es l'AGENT GIT. Tu inspectes le dépôt git de {WORKDIR} "
            "avec tes outils MCP git (status, diff, log, add, commit). "
            "Réponds en français en rapportant fidèlement la sortie des outils."),
)

analyse_agent = create_react_agent(
    llm, custom_tools,
    prompt=("Tu es l'AGENT ANALYSE. Tu utilises les outils du serveur MCP "
            "personnalisé custom_ops : summarize_lines (statistiques), "
            "format_markdown (rapports), generate_changelog (changelog depuis un diff). "
            "Réponds en français."),
)

# --- Envelopper chaque agent comme un OUTIL pour le superviseur ---
async def _call(agent, instruction: str) -> str:
    result = await agent.ainvoke({"messages": [("user", instruction)]})
    return result["messages"][-1].content

@tool
async def agent_fichiers(instruction: str) -> str:
    """Délègue une tâche à l'agent FICHIERS (serveur MCP filesystem) :
    lister, lire, écrire ou créer des fichiers dans l'espace de travail."""
    print(f"  📁 [superviseur → agent_fichiers] {instruction}")
    return await _call(fichiers_agent, instruction)

@tool
async def agent_git(instruction: str) -> str:
    """Délègue une tâche à l'agent GIT (serveur MCP git) :
    statut, diff, historique, commits du dépôt."""
    print(f"  🌿 [superviseur → agent_git] {instruction}")
    return await _call(git_agent, instruction)

@tool
async def agent_analyse(instruction: str) -> str:
    """Délègue une tâche à l'agent ANALYSE (serveur MCP custom_ops) :
    statistiques de texte, mise en forme Markdown, génération de changelog.
    Fournis-lui les données brutes (texte, diff) dans l'instruction."""
    print(f"  📊 [superviseur → agent_analyse] {instruction}")
    return await _call(analyse_agent, instruction)

# --- Le SUPERVISEUR : Gemini + les 3 agents-outils ---
superviseur = create_react_agent(
    llm,
    [agent_fichiers, agent_git, agent_analyse],
    prompt=(
        "Tu es le SUPERVISEUR d'une équipe de 3 agents spécialisés :\n"
        "1. agent_fichiers → tout ce qui concerne les fichiers (lire/écrire/lister)\n"
        "2. agent_git → tout ce qui concerne le dépôt git (status/diff/log/commit)\n"
        "3. agent_analyse → statistiques, formatage Markdown, changelogs\n\n"
        "À chaque étape, DÉCIDE toi-même quel agent appeler et avec quelle "
        "instruction précise, en fonction de ce qu'il reste à faire. Tu peux "
        "enchaîner plusieurs appels et transmettre le résultat d'un agent à un "
        "autre (ex. donner un diff git à agent_analyse). Quand la tâche complète "
        "est terminée, rédige une synthèse finale en français."
    ),
)

print("✅ Système multi-agents prêt (superviseur + 3 agents)")

### 🚀 Démo finale — Tâche complexe orchestrée de bout en bout

Une seule requête utilisateur → le superviseur décompose, délègue et combine **librement** :
il devra probablement appeler `agent_git` (diff), puis `agent_analyse` (changelog + rapport), puis `agent_fichiers` (écrire le rapport) — mais **c'est lui qui décide**, rien n'est scripté.

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(run_agent(
    superviseur,
    "Fais un audit complet de l'espace de travail :\n"
    "1) Récupère la liste des fichiers et le contenu de TODO.txt ;\n"
    "2) Obtiens des statistiques sur les lignes de TODO.txt ;\n"
    "3) Récupère le diff git des modifications non commitées et génère un changelog ;\n"
    "4) Assemble le tout dans un rapport Markdown et écris-le dans RAPPORT.md ;\n"
    "5) Termine par une synthèse."
))

In [ ]:
# Vérification : le rapport a-t-il bien été écrit par l'agent ?
rapport = Path(WORKDIR) / "RAPPORT.md"
if rapport.exists():
    print("✅ RAPPORT.md créé par le système multi-agents :\n")
    print(rapport.read_text(encoding="utf-8"))
else:
    print("⚠️ RAPPORT.md non trouvé — relancez la démo ou inspectez la trace ci-dessus.")

### 🚀 Démo bonus — La politique est vraiment flexible

Une requête différente → un plan différent, choisi par le LLM (ici il n'a pas besoin de git).

In [ ]:
_ = asyncio.get_event_loop().run_until_complete(run_agent(
    superviseur,
    "Lis app.py, explique ce que fait le code, donne des statistiques sur ses lignes, "
    "puis propose une amélioration."
))

---
## ✅ Conclusion — Ce que ce projet démontre

| Exigence | Réalisation |
|---|---|
| Environnement Colab | ✅ Notebook 100 % Colab (Node installable via `apt-get`) |
| ≥ 2 serveurs MCP **tiers** | ✅ `@modelcontextprotocol/server-filesystem` (npx) + `mcp-server-git` (python) |
| Transport **stdio** | ✅ Tous les serveurs sont lancés en sous-processus stdio par `MultiServerMCPClient` |
| Serveur MCP **personnalisé** | ✅ `custom_ops` en FastMCP (`ping`, `summarize_lines`, `format_markdown`, `generate_changelog`) |
| Agent **Gemini** | ✅ `ChatGoogleGenerativeAI` (gemini-2.5-flash) + boucle ReAct LangGraph |
| Application **multi-agents** | ✅ Superviseur + 3 agents spécialisés (un par serveur MCP) |
| Politique **pilotée par outil** | ✅ Aucun flux prédéfini : le superviseur choisit dynamiquement les agents (et chaque agent choisit ses outils MCP) via les *tool calls* du LLM |

**Pistes d'extension :**
* Ajouter d'autres serveurs MCP tiers (ex. `server-fetch` pour le web, `server-memory` pour la mémoire) ;
* Ajouter une mémoire de conversation (`checkpointer` LangGraph) pour un assistant persistant ;
* Exposer le superviseur derrière une petite UI (Gradio) directement dans Colab.